In [3]:
import os

os.chdir("..")
import torch
import sys
torch.set_printoptions(threshold=sys.maxsize, linewidth=sys.maxsize)

In [ ]:
from PIL import Image
import torchvision.transforms.functional as TF
from matplotlib import pyplot as plt

im = Image.open("/scratch/bdej/cohanlon/unlabeled/train/rgb/million-case/220683.jpg")
im1 = TF.rotate(im, 45.4, Image.Resampling.NEAREST)
im2 = TF.rotate(im, 45.4, Image.Resampling.BILINEAR)
im3 = TF.rotate(im, 45.4, Image.Resampling.BICUBIC)

plt.figure()
f, axarr = plt.subplots(1, 3)
axarr[0].imshow(im1)
axarr[1].imshow(im2)
axarr[2].imshow(im3)

plt.show()

In [ ]:
from src.data.modality_transforms import RGBTransform
from src.data.modality_info import MODALITY_INFO
import torch


def setup_modality_info(args):
    """Sets up the modality info dictionary for the given domains."""
    modality_info = {mod: MODALITY_INFO[mod] for mod in args}
    return modality_info


modality_info = setup_modality_info(["rgb"])
modality_paths = {"rgb": "rgb"}

MODALITY_TRANSFORMS_VQVAE = {}
MODALITY_TRANSFORMS_VQVAE["rgb"] = RGBTransform(
    mean_and_std="naip", color_jitter=False, no_data_value=0
)

image_augmenter_train = RandomRotationImageAugmenter(near_orthogonal=True)

transforms_train = UnifiedDataTransform(
    transforms_dict=MODALITY_TRANSFORMS_VQVAE,
    image_augmenter=image_augmenter_train,
    resample_mode="bicubic",
    add_sizes=False,
)

dataset_train = MultiModalDatasetFolder(
    root="/scratch/bdej/cohanlon/unlabeled/train",
    modalities=["rgb", "mask_valid"],
    modality_paths=modality_paths,
    modality_transforms=MODALITY_TRANSFORMS_VQVAE,
    modality_info=modality_info,
    transform=transforms_train,
    cache=False,
)

sampler_train = torch.utils.data.DistributedSampler(
    dataset_train,
    num_replicas=1,
    rank=0,
    shuffle=True,
    drop_last=True,
)

data_loader_train = torch.utils.data.DataLoader(
    dataset_train,
    sampler=sampler_train,
    batch_size=64,
    num_workers=1,
    pin_memory=False,
    drop_last=True,
)

In [ ]:
import time

start = time.time()
for i in range(4):
    next(iter(data_loader_train))
print(time.time() - start)


In [ ]:
dataset_train.class_to_idx

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from src.data.modality_transforms import DepthTransform

depth_transform = DepthTransform(no_data_value=-9999.0)

# im = Image.open(
#     "/scratch/bdej/cohanlon/data/train/depth/million-case/DEM_270091.tif"
# )
im = Image.open(
    "/scratch/bdej/cohanlon/data/train/depth/million-case/DEM_901.tif"
)
t = depth_transform.image_augment(im, None, None, 0, None, None, None, "bilinear")
t = depth_transform.postprocess(t)
t = depth_transform.depth_minmax_scaling(t)

t_prime = t.flatten()
mean = t_prime.mean().item()
std = t_prime.std().item()

print(t_prime.mode()[0].item(), t_prime.min().item(), t_prime.max().item(), t_prime.mean().item(), t_prime.std().item())
plt.hist(t.flatten(), log=True, bins=100)
# t = depth_transform.truncated_depth_standardization(t)
plt.show()

In [ ]:
from scipy.stats import norm
plt.plot(np.linspace(-178, 178, 100), norm.pdf(np.linspace(-178, 178, 100), loc=mean, scale=std))
plt.show()

In [ ]:
from PIL import Image
import numpy as np
import torchvision.transforms.functional as TF
im = Image.open("/scratch/bdej/cohanlon/data/train/rgb/million-case/90181.jpg")
TF.to_tensor(im).shape

In [ ]:
import numpy as np
import torch

test = np.array([[-np.inf, 2, 3], [4, 5, np.inf], [-9999.0, np.nan, 9]])
test = np.expand_dims(test, 0)
test

In [ ]:
test = torch.tensor(test)
test

In [ ]:
no_data_mods = ["depth"]
sample_dict = {"depth": test}
modality_info = {"depth": {"no_data_value": -9999.0, "num_channels": 1}}

channel_dim = 0
H, W = sample_dict[no_data_mods[0]].shape[1:3]
mask = torch.ones(1, H, W, dtype=torch.bool)
for mod in no_data_mods:
    sample = sample_dict[mod]
    assert sample.shape[channel_dim] == modality_info[mod]["num_channels"]
    no_data_value = modality_info[mod]["no_data_value"]

    no_data_mask = (sample != no_data_value).all(dim=channel_dim, keepdim=True)
    nan_mask = np.isfinite(sample).any(dim=channel_dim, keepdim=True)
    mask = mask & no_data_mask & nan_mask

mask

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from scipy.stats import norm

arr = np.load("../clean_images.npy")
arr = arr[:, 0].flatten()
arr = arr[arr != 0]
mean = arr.mean()
std =  arr.std()


# def reject_outliers(data, m=2.0):
#     d = np.abs(data - np.median(data))
#     mdev = np.median(d)
#     s = d / mdev if mdev else np.zeros(len(d))
#     return data[s < m]

# arr = reject_outliers(arr, 3)

plt.hist(arr, bins=1000, log=True, density=False)
plt.show()
arr.mean(), arr.std(), arr.min(), arr.max()

In [ ]:
arr_trim = arr[(arr > mean - 3 * std) & (arr < mean + 3 * std)]

plt.hist(arr_trim, bins=1000, log=True, density=False)
plt.show()
arr_trim.mean(), arr_trim.std(), arr_trim.min(), arr_trim.max()

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from PIL import Image
from scipy.stats import mode, iqr
import torch
arr = np.array(Image.open("/scratch/bdej/cohanlon/data/train/depth/million-case/DEM_79.tif"))

from src.data.modality_transforms import DepthTransform

arr = np.expand_dims(arr, axis=0)

arr = arr.transpose(1, 2, 0)
arr = torch.tensor(arr)

mask = DepthTransform.depth_artifact_mask(arr, outlier_threshold=16)

# arr = DepthTransform.depth_robust_scaling(arr)

arr = DepthTransform.depth_minmax_scaling(arr)

arr[~mask] = 0

# # Robust scaling
# arr[arr == -9999.0] = 0
# arr = (arr - np.median(arr[arr != 0])) / iqr(arr[arr != 0])

# # Robust trimming
# dists_from_median = np.abs(arr - np.median(arr[arr != 0]))
# dists_in_iqrs = dists_from_median / iqr(arr[arr != 0])
# outlier_threshold = 7
# # make removed pix (the outliers) black
# arr[dists_in_iqrs > outlier_threshold] = 0

d_arr = arr[arr != 0].numpy().flatten()
print(mode(d_arr).mode, d_arr.mean(), np.median(d_arr), d_arr.std(), d_arr.min(), d_arr.max())
plt.hist(d_arr, bins=1000, log=True, density=False)
plt.show()

# make kept pix white
# arr[dists_in_iqrs <= outlier_threshold] = 1

# arr[mask] = 1

print("Remaining:", round(len(arr[mask]) / 224 ** 2 * 100, 2), "%") 

arr = (arr.numpy() * 255).astype(np.uint8).repeat(3, axis=2)
im = Image.fromarray(arr)
im

In [ ]:
import numpy as np
import torch
import os
from pathlib import Path

root = Path("/scratch/bdej/cohanlon/data/train/rgb_toks/million-case")

arr = np.load(root / "200683.npy")
arr

In [ ]:
import torch
from einops import repeat
tokens = torch.arange(0, 16).reshape(4, 4).unsqueeze(0)
tokens

In [ ]:
mask = torch.tensor([[0, 1, 1, 0]])
mask_arange = (torch.arange(0, 4) * 1e-6).unsqueeze(0)
ids_shuffle = torch.argsort(mask + mask_arange, dim=1)[:, :4]
print(ids_shuffle)
# torch.gather(tokens, dim=1, index=repeat(ids_shuffle, "b n -> b n d", d=tokens.shape[-1]))
# ids = repeat(ids_shuffle, "b n -> b n d", d=tokens.shape[-1])
tokens[:, [3, 0], :]

In [ ]:
import os

os.chdir("..")
import torch
import torch.nn as nn
from src.models.unet import PatchedConvNeXtUNet

# TODO: we need to control for params and depth
# should be near patched unet base
model = PatchedConvNeXtUNet(
    in_channels=1,
    out_channels=1,
    num_heads=8,
    cond_dim=768,
    act_layer=nn.SiLU,
    patch_size=8,
    model_channels=128,
    num_conv_blocks=3,
    channel_mult=(1, 2, 4, 8, 16),
)
print(f"# params: {sum(p.numel() for p in model.parameters()):_}")
model

In [ ]:
x = torch.randn(5, 1, 224, 224)
timesteps = torch.randint(10, size=(5,))
cond = torch.randn(5, 1, 768)
out = model(x, timesteps, cond)
out.shape

In [ ]:
from src.models.sarformer import sarformer_t_swiglu_qknorm_nobias

model = sarformer_t_swiglu_qknorm_nobias(encoder_embeddings=None)
print(f"{model.get_num_encoder_params():_}, {model.get_num_backbone_params():_}")

In [ ]:
import torch
from src.models.encoder_embeddings import ImageTokenEncoderEmbedding
from src.models.sarformer import sarformer_t_swiglu_qknorm
from src.data.modality_info import MODALITY_INFO

encoder_embeddings = {
    "tok_rgb@224": ImageTokenEncoderEmbedding(
        vocab_size=16384, patch_size=16, dim_tokens=768, image_size=224
    )
}

mod_dict = {"tok_rgb@224": {"tensor": torch.randint(16385, size=(5, 14, 14))}}
noisy_image = torch.randn(5, 1, 224, 224)
timesteps = torch.randint(10, size=(5,))

model = sarformer_t_swiglu_qknorm(
    encoder_embeddings=encoder_embeddings,
    modality_info=MODALITY_INFO,
)
print(model.get_num_encoder_params())
model

In [ ]:
from tokenizers import Tokenizer
from copy import deepcopy
import numpy as np
tokenizer = Tokenizer.from_file("../fourm/utils/tokenizer/trained/tokenizer_inc_nonUS_lower.json")

tokenizer.enable_padding(length=512)
tokenizer.enable_truncation(max_length=512)

In [ ]:
import random
import torch
import numpy as np
from PIL import Image
import torch.nn.functional as F
from einops import rearrange, repeat


def spatial_softmax(x):
    H, W = x.shape[-2:]
    x = rearrange(x, "... c h w -> ... c (h w)")
    x = F.softmax(x, dim=-1)
    return rearrange(x, "... c (h w) -> ... c h w", h=H, w=W)


def create_overlaid_img(
    target_dist: torch.Tensor, pred_dist: torch.Tensor, original_img_path: str
):
    """Creates an image with the target and predictions overlaid on the original image.
    The target distribution's single non-zero value is replaced by a black square to make it more visible.
    All distributions should sum to 1.

    Args:
        target_dist: The target distribution. Shape (1, H, W)
        pred_dist: The predicted distribution. Shape (1, H, W)
        original_img_path: Path to the original image.
    """
    pred_alpha_mask = (
        255 * rearrange(pred_dist, "1 h w -> h w 1").float().cpu().numpy()
    ).astype(np.uint8)

    H, W = target_dist.shape[-2:]

    # Byte array that corresponds to a yellow image
    yellow = np.concatenate(
        [255 * np.ones((H, W, 2)), np.zeros((H, W, 1))], axis=-1
    ).astype(np.uint8)

    pred_yellow_img = Image.fromarray(
        np.concatenate([yellow, pred_alpha_mask], axis=-1), mode="RGBA"
    )

    # White image with completely transparent alpha channel
    target_img_arr = np.concatenate(
        [255 * np.ones((H, W, 3)), np.zeros((H, W, 1))], axis=-1
    ).astype(np.uint8)

    # (i, j) of the single non-zero value in the target distribution
    i = np.argmax(target_dist, axis=-2).max().item()
    j = np.argmax(target_dist, axis=-1).max().item()

    margin = 1  # Pixel margin for black square
    lower_i = i - (margin if i - margin >= 0 else 0)
    upper_i = i + (1 + margin if i + margin < H else 1)
    lower_j = j - (margin if j - margin >= 0 else 0)
    upper_j = j + (1 + margin if j + margin < W else 1)

    # Create black square at the position of the 1 in the target distribution
    target_img_arr[lower_i:upper_i, lower_j:upper_j, :-1] = 0
    # Set the black square to full opacity
    target_img_arr[lower_i:upper_i, lower_j:upper_j, -1] = 255

    target_img = Image.fromarray(target_img_arr, mode="RGBA")

    im = Image.open(original_img_path)
    im.paste(pred_yellow_img, (0, 0), pred_yellow_img)
    im.paste(target_img, (0, 0), target_img)

    return im


t = torch.zeros(1, 224, 224)
i, j = random.randint(0, 223), random.randint(0, 223)
t[:, i, j] = 1

p = torch.randn(1, 224, 224)
# p = spatial_softmax(p)

create_overlaid_img(
    t, p, "/scratch/bdej/cohanlon/data/train/rgb/labeled_sar/NYLabeled1451.tif"
)

In [ ]:
import torch
from einops import repeat

t = torch.tensor([[0, 0], [0, 1]])
a = t.argmax(dim=-2).max().item()
b = t.argmax(dim=-1).max().item()
print(a, b)

# H, W = 2, 2
# i = repeat(torch.arange(H), "h -> 1 h w", w=W)
# j = repeat(torch.arange(W), "w -> 1 h w", h=H)

# dists = torch.sqrt((i - a) ** 2 + (j - b) ** 2)
# dists.shape


In [ ]:
import os

os.chdir("..")
from run_training_sarformer import distance_weighted_loss
import torch

logits = torch.randn(1, 1, 224, 224)
target = torch.zeros_like(logits)
target[:, :, 112, 112] = 1

distance_weighted_loss(logits, target, "bce", "euclidean")

In [ ]:
import os

os.chdir("..")
from src.data.multimodal_dataset_folder import MultiModalDatasetFolder
from torch.utils.data import DataLoader, SequentialSampler
from src.data.modality_info import MODALITY_TRANSFORMS, MODALITY_INFO
from src.data.image_augmenter import CropImageAugmenter
from src.data.modality_transforms import MaskTransform, TargetDistributionTransform, UnifiedDataTransform
from argparse import Namespace

args = Namespace()
args.data_path = "/scratch/bdej/cohanlon/data/train"
args.all_domains = ["rgb", "depth", "mask", "caption"]  # add structured when available
args.batch_size = 2
args.num_workers = 0
args.pin_mem = False
args.full_img_size = 673
args.eff_img_size = 447
args.target_size = 224
args.mask_proportion = 0.6
args.patch_size = 32
args.crop_std = 0.1
args.hflip = 0.0
args.vflip = 0.0

modality_info = MODALITY_INFO
modality_transforms = MODALITY_TRANSFORMS
if "target_distribution" in args.all_domains:
    modality_transforms["target_distribution"] = TargetDistributionTransform(
        img_size=args.full_img_size
    )
if "mask" in args.all_domains:
    modality_transforms["mask"] = MaskTransform(
        mask_size=args.target_size,
        mask_proportion=args.mask_proportion,
        patch_size=args.patch_size,
    )
transform = UnifiedDataTransform(
    transforms_dict=modality_transforms,
    image_augmenter=CropImageAugmenter(
        img_size=args.full_img_size,
        eff_img_size=args.eff_img_size,
        target_size=args.target_size,
        random_crop_std=args.crop_std,
        hflip=args.hflip,
        vflip=args.vflip,
    ),
)

dataset_train = MultiModalDatasetFolder(
    root=args.data_path,
    modalities=args.all_domains,
    modality_transforms=modality_transforms,
    modality_info=modality_info,
    transform=transform,
)
print("dataset_train size = %d" % len(dataset_train))

sampler_train = SequentialSampler(dataset_train)

data_loader_train = DataLoader(
    dataset_train,
    sampler=sampler_train,
    batch_size=args.batch_size,
    num_workers=args.num_workers,
    pin_memory=args.pin_mem,
    drop_last=True,
)

In [3]:
import torchvision.transforms.functional as TF
from PIL import Image

im = Image.open("/scratch/bdej/cohanlon/data/train/rgb/sar/AZLabeled123.tif")

In [ ]:
from scipy.stats import truncnorm

img_size = 673
eff_img_size = 673
crop_size = 224
std = 1e12
start = (img_size - eff_img_size) // 2
end = img_size - start - crop_size

mean = (start + end) / 2

# compute bounds of truncated normal distribution in stds
lower_std = (start - mean) / std
upper_std = (end - mean) / std

# sample truncated normal distribution and round to discretize it
top = round(
    truncnorm.rvs(lower_std, upper_std, loc=mean, scale=std)
)
left = round(
    truncnorm.rvs(lower_std, upper_std, loc=mean, scale=std)
)
print(start, end, end - start + crop_size)
print(top, left, mean, std)
TF.crop(im, top, left, crop_size, crop_size)

In [ ]:
import os
os.chdir("..")
from src.models.sarformer import sarformer_s
import torch

checkpoint = torch.load(
    "/scratch/bdej/cohanlon/checkpoints/sarformer/model/pretrain/S_MAE_rgb-depth-caption/checkpoint-299.pth",
    map_location="cpu",
)
model = sarformer_s(in_channels=4, out_channels=1, is_pretraining=False)

In [ ]:
[key for key in checkpoint["model"].keys() if "out_proj" in key]
# model.load_state_dict(pretrained_weights, strict=False)

In [ ]:
model.state_dict()["backbone.out_proj.weight"].shape

In [3]:
from transformers import ConvNextConfig, ConvNextModel
import torch

img = torch.randn((1, 4, 224, 224))
config = ConvNextConfig(num_channels=4)
model = ConvNextModel(config)
test_input = torch.randn(1, 4, 224, 224)

In [ ]:
import torch
from einops import repeat
import torch.nn.functional as F


def spatial_softmax(x):
    """Apply softmax to spatial dimensions of the input tensor.

    Args:
        x: Input image tensor. Shape (B, C, H, W)
    """
    # Flatten spatial dims, apply softmax, restore shape
    x = F.softmax(x.contiguous().flatten(-2), dim=-1).view_as(x)
    return x


def distance_weighted_loss(
    logits,
    target,
    loss_type="mse",
    distance_type="euclidean",
    reduction="mean",
):
    """
    Computes the distance-weighted loss between the predicted spatial
    distribution and the target one-hot distribution.

    Args:
        logits:  spatial distribution. Shape (B, C, H, W) or (B, 2) for regression on point.
        target: Target one-hot distribution. Shape (B, C, H, W)
        loss_type: Type of loss to use. Supported types are "bce" (binary cross-entropy)
                "ce" (cross-entropy) and "mse" (mean-squared error).
        distance_type: Type of distance to use. Supported types are "euclidean", "manhattan" or "none".
        eps: Small constant to avoid division by zero.
        reduction: Reduction type for the loss. Supported types are "mean", "sum", and "none".
    """
    if loss_type == "none" and logits.ndim != 2:
        raise ValueError(
            f"Loss function expects input for regression on point (B, 2). Got {logits.shape}"
        )
    elif loss_type != "none" and logits.ndim != 4:
        raise ValueError(
            f"Loss function expects spatial distribution (B, C, H, W). Got {logits.shape}"
        )
    with torch.no_grad():
        # Index positions of the single non-zero value in the target distribution
        target_ys = target.argmax(dim=2).max(dim=-1)[0].squeeze(-1).float()
        target_xs = target.argmax(dim=3).max(dim=-1)[0].squeeze(-1).float()

        if loss_type != "none":
            B, _, H, W = logits.shape
            device = logits.device
            # Row and column indice tensors
            i = repeat(torch.arange(H, device=device), "h -> b 1 h w", b=B, w=W)
            j = repeat(torch.arange(W, device=device), "w -> b 1 h w", b=B, h=H)
            target_ys = repeat(target_ys, "b -> b 1 h w", h=H, w=W)
            target_xs = repeat(target_xs, "b -> b 1 h w", h=H, w=W)

            if distance_type == "euclidean":
                dists = torch.sqrt((i - target_ys) ** 2 + (j - target_xs) ** 2)
            elif distance_type == "manhattan":
                dists = torch.abs(i - target_ys) + torch.abs(j - target_xs)
            elif distance_type == "none":
                dists = torch.ones_like(logits, device=device)  # (B, C, H, W)
            else:
                raise ValueError(f"Unsupported distance type: {distance_type}")
        else:
            target_coords = torch.stack([target_ys, target_xs], dim=1)

    if loss_type != "none":
        max_dists = dists.flatten(-2).max(dim=-1).values[:, None, None]
        dists = dists / max_dists + 1
        pred = spatial_softmax(logits)

        if loss_type == "bce":
            loss = -dists * (
                target * torch.log(pred) + (1 - target) * torch.log(1 - pred)
            )
        elif loss_type == "ce":
            loss = -dists * target * torch.log(pred)
        elif loss_type == "mse":
            loss = dists * F.mse_loss(pred, target, reduction="none")
        else:
            raise ValueError(f"Unsupported loss type: {loss_type}")
        loss = loss.sum(dim=(1, 2, 3))  # Sum over channel and spatial dimensions
    else:
        loss = torch.norm(target_coords - logits, dim=1)  # l2 norm (by default)

    # Reduce over batch dimension
    if reduction == "mean":
        return loss.mean()
    elif reduction == "sum":
        return loss.sum()
    elif reduction == "none":
        return loss
    else:
        raise ValueError(f"Unsupported reduction type: {reduction}")


target = (
    torch.tensor([[[0, 0, 0], [0, 1, 0], [0, 0, 0]], [[0, 0, 0], [0, 0, 0], [0, 0, 1]], [[1, 0, 0], [0, 0, 0], [0, 0, 0]]])
    .unsqueeze(1)
    .float()
)
logits = (
    torch.tensor([[[0, 0, 0], [1e10, 0, 0], [0, 0, 0]], [[0, 0, 0], [0, 1e10, 0], [0, 0, 0]], [[1e10, 0, 0], [0, 0, 0], [0, 0, 0]]])
    .unsqueeze(1)
    .float()
)
distance_weighted_loss(logits, target, "mse", "euclidean")

In [1]:
import os
os.chdir("..")
import torch
from src.models.point_regression_model import convnext_gatedmlp_t
model = convnext_gatedmlp_t(in_channels=1)
input = torch.randn((2, 1, 224, 224))


/u/cohanlon/micromamba/envs/sarformer/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch
dct = torch.load(
    "/scratch/bdej/cohanlon/checkpoints/sarformer/model/misc/B_ConvNeXt_GatedMLP_regrpoint_depth_2/debug_mod_dict.pt"
)
target_dist = dct["target_distribution"]
depth = dct["depth"]
output = model(depth)

/tmp/ipykernel_1706118/1729149274.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  dct = torch.load(


In [3]:
import os
os.chdir("/u/cohanlon/sarformer")
from run_training_sarformer import distance_weighted_loss

loss = lambda logits, target: distance_weighted_loss(logits, target, loss_type="none")
loss = loss(output, target_dist)
loss

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


tensor(169.3293, grad_fn=<MeanBackward0>)

In [ ]:
import torch
output = torch.load(
    "/scratch/bdej/cohanlon/checkpoints/sarformer/model/misc/B_ConvNeXt_GatedMLP_regrpoint_depth_2/debug_mod_dict.pt"
)
depth = output["depth"]
depth

/tmp/ipykernel_486024/1390234861.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  output = torch.load(


tensor([[[[0.0329, 0.0333, 0.0338,  ..., 0.0413, 0.0449, 0.0482],
          [0.0317, 0.0322, 0.0328,  ..., 0.0387, 0.0431, 0.0471],
          [0.0304, 0.0312, 0.0319,  ..., 0.0366, 0.0407, 0.0449],
          ...,
          [0.0653, 0.0644, 0.0635,  ..., 0.0770, 0.0780, 0.0787],
          [0.0651, 0.0645, 0.0639,  ..., 0.0780, 0.0788, 0.0796],
          [0.0648, 0.0646, 0.0643,  ..., 0.0791, 0.0797, 0.0806]]],


        [[[0.0970, 0.0972, 0.0973,  ..., 0.0743, 0.0742, 0.0741],
          [0.0969, 0.0971, 0.0973,  ..., 0.0739, 0.0738, 0.0737],
          [0.0969, 0.0971, 0.0972,  ..., 0.0735, 0.0735, 0.0734],
          ...,
          [0.0008, 0.0008, 0.0007,  ..., 0.0002, 0.0002, 0.0002],
          [0.0007, 0.0007, 0.0007,  ..., 0.0002, 0.0002, 0.0002],
          [0.0007, 0.0007, 0.0007,  ..., 0.0002, 0.0002, 0.0002]]],


        [[[0.0081, 0.0092, 0.0103,  ..., 0.0989, 0.0994, 0.1000],
          [0.0075, 0.0085, 0.0095,  ..., 0.0982, 0.0988, 0.0994],
          [0.0069, 0.0077, 0.0087,  ..